This notebook can be used to verify the dataframes displayed values in Napari. 

In [ ]:
import napari
import pandas as pd
from pathlib import Path
import numpy as np
from skimage import io
viewer = napari.Viewer()
pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv('test_data.csv')
df = df.drop('Unnamed: 0',axis=1)

Construct unique identifier used to select cells for displaying in Napari.

In [ ]:
df['fov']=df['source_folder'].str[-2:]
df['uid'] = (
    df['strain'].astype(str) + '_' +
    df['replicate'].astype(str) + '_' +
    df['image_cell_id'].astype(str) + '_' +
    df['fov'].astype(str)
)

In [ ]:
#showcase only relevant columns
showcase_df = df[['uid','image_cell_id', 'mRNA', 'area',
       'area_bud', 'spots', 'dense_regions', 'decomposed_RNAs',
       'tx', 'nascent_RNAs', 'total_RNAs', 'spots_bud', 'dense_regions_bud',
       'decomposed_RNAs_bud', 'tx_bud', 'nascent_RNAs_bud',
       'total_RNAs_bud', 'mother_nuclei_count','spots_mother', 'total_RNAs_mother',
      'replicate','condition + strain']]

Look for dataframe entries to visualize in Napari.

In [ ]:
showcase_df[showcase_df['dense_regions']>10]

Use uid to select cell.

In [ ]:
uid = 'YET572_BR4_33_03'	
rows = df[df['uid']==uid]
basepath = Path(rows['source_folder'].values[0])

# Get bounding box coordinates
y0 = int(rows['bbox-0'].iloc[0])
x0 = int(rows['bbox-1'].iloc[0])
y1 = int(rows['bbox-2'].iloc[0])
x1 = int(rows['bbox-3'].iloc[0])
showcase_df[showcase_df['uid']==uid]

Show cell in napari.

In [ ]:
spot_paths = list(basepath.glob('*spots.npy'))
masks_paths = list(Path(basepath,'corrected').glob('*.tif'))

ASH1CLB2 = np.amax(io.imread(Path(basepath,'ASH1CLB2.tif')),axis=0)
SRL1 = np.amax(io.imread(Path(basepath,'SRL1.tif')),axis=0)
SUN4 = np.amax(io.imread(Path(basepath,'SUN4.tif')),axis=0)
DAPI = np.amax(io.imread(Path(basepath,'DAPI.tif')),axis=0)

# Define rectangle as 2D array: [[y0, x0], [y0, x1], [y1, x1], [y1, x0]]
rectangle = [[y0, x0], [y0, x1], [y1, x1], [y1, x0]]
viewer.add_shapes([rectangle], shape_type='rectangle', edge_color='red',blending='additive',edge_width=4, face_color='transparent', name=f'cell_{uid}')

for spot_path in spot_paths[:3]:
    name = spot_path.name.split('_')[0] 
    viewer.add_points(np.load(spot_path)[:,1:3],name=spot_path.stem, visible=False, face_color='blue')
    if name == 'ASH1CLB2':
        viewer.add_image(ASH1CLB2, name='ASH1CLB2', blending='additive', visible=False, contrast_limits=[0,12000])
    elif name == 'SRL1':
        viewer.add_image(SRL1, name='SRL1', blending='additive', visible=False, contrast_limits=[0,12000])
    elif name == 'SUN4':
        viewer.add_image(SUN4, name='SUN4', blending='additive', visible=True ,contrast_limits=[0,12000])

viewer.add_image(DAPI, name='DAPI', blending='additive', visible=False ,contrast_limits=[0,25000])
        
for masks_path in masks_paths:
    viewer.add_labels(io.imread(masks_path),name=masks_path.stem)

Clear all layers.

In [ ]:
viewer.layers.clear()